# OpenStream — Hate Speech Classifier v4

**BERTweet + LoRA (r=32, query+key+value+dense) + Focal Loss (γ=1) + Cosine LR with Warmup + Threshold-Tuned Checkpoint Selection**

This notebook keeps the strong v3 training configuration while aligning preprocessing with BERTweet's tokenizer normalization and selecting the best checkpoint using the same threshold-tuned validation Macro F1 used at deployment.

The held-out test set is used only once for final evaluation. Threshold selection and checkpoint selection use validation data only.


## 1. Environment

In [ ]:
!pip install -q transformers peft torch scikit-learn pandas matplotlib seaborn emoji

In [ ]:
!pip install -q --upgrade "torchao>0.16.0"

In [ ]:
import os, re, json, urllib.request, warnings, shutil
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
    logging as hf_logging,
)
hf_logging.set_verbosity_error()

from peft import get_peft_model, LoraConfig, TaskType, PeftModel

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns


## 2. Configuration

In [ ]:
RANDOM_SEED   = 42
MODEL_NAME    = "vinai/bertweet-base"

MAX_LENGTH    = 128
BATCH_SIZE    = 32
NUM_LABELS    = 2

# Stability-focused experiment configuration
LORA_R        = 32
LORA_ALPHA    = 64
LORA_DROPOUT  = 0.20
LORA_TARGETS  = ["query", "key", "value", "dense"]

FOCAL_GAMMA   = 1.0
LEARNING_RATE = 5e-5
WEIGHT_DECAY  = 0.05
NUM_EPOCHS    = 5
PATIENCE      = 2
WARMUP_RATIO  = 0.06

# Manual class weighting: substantially lighter minority weighting than the previous ~3x ratio
CLASS_WEIGHTS = torch.tensor([1.0, 1.25], dtype=torch.float)

RAW_DIR       = "data/raw"
PROCESSED_DIR = "data/processed_v4"
OUT_DIR       = "outputs/bertweet_lora32_focal1_lr5e5_warmup_v4"
ARTIFACT_DIR  = "artifacts/openstream-moderation-lora32-focal1-v4"
ARCHIVE_NAME  = "openstream-moderation-model-v4"

LABEL2ID = {"normal": 0, "flagged": 1}
ID2LABEL = {0: "normal", 1: "flagged"}

for d in [RAW_DIR, PROCESSED_DIR, OUT_DIR, ARTIFACT_DIR]:
    os.makedirs(d, exist_ok=True)

CLASS_WEIGHTS = CLASS_WEIGHTS.to("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")


## 3. Data

Jigsaw Toxic Comment Classification training data. The six toxicity-related labels are collapsed into the existing binary setup: `normal=0`, `flagged=1`.


In [ ]:
# GITHUB_RAW = "https://raw.githubusercontent.com/mohaimeed/THOS/main"
# TEXT_FILE  = os.path.join(RAW_DIR, "THOS_Dataset_Text.json")
# LABEL_FILE = os.path.join(RAW_DIR, "THOS_Dataset.json")

# def download_file(url: str, dest: str):
#     if os.path.exists(dest):
#         print(f"cached: {dest}")
#         return
#     print(f"downloading {url}")
#     urllib.request.urlretrieve(url, dest)

# download_file(f"{GITHUB_RAW}/THOS_Dataset_Text.json", TEXT_FILE)
# download_file(f"{GITHUB_RAW}/THOS_Dataset.json", LABEL_FILE)

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np

# 1. Load a clean, script-free Parquet mirror of the exact same Jigsaw dataset
dataset = load_dataset("thesofakillers/jigsaw-toxic-comment-classification-challenge")
df = dataset["train"].to_pandas()

# 2. Map labels strictly to binary setup (normal: 0, flagged: 1)
# Includes generic 'toxic' to prevent data poisoning
is_violating = (
    (df['toxic'] == 1) |
    (df['threat'] == 1) |
    (df['identity_hate'] == 1) |
    (df['severe_toxic'] == 1) |
    (df['insult'] == 1) |
    (df['obscene'] == 1)
)

df['label'] = np.where(is_violating, LABEL2ID['flagged'], LABEL2ID['normal'])
df = df.rename(columns={'comment_text': 'text'})[['text', 'label']]

### 3.1 Label derivation

A comment is `flagged` if any of `toxic`, `threat`, `identity_hate`, `severe_toxic`, `insult`, or `obscene` is positive. Otherwise it is `normal`.


In [ ]:
# The previous cell already derived the labels directly into 'label' and dropped original columns.
# Legacy derivation logic is skipped to avoid KeyErrors.
print("Labels are already derived and mapped in the dataset. Skipping.")

### 3.2 BERTweet-aligned text preparation

Keep the source text largely intact and let `AutoTokenizer(..., normalization=True)` perform BERTweet's tweet-style normalization. Do not lowercase globally, strip hashtags, or substitute custom `[USER]` / `[URL]` tokens before tokenization.


In [ ]:
# BERTweet-aligned text preparation.
# Keep lexical/case/hashtag information intact and let the BERTweet tokenizer
# perform its own normalization (e.g. mentions/URLs) via normalization=True.
print("Preparing text for BERTweet...")

df["text"] = (
    df["text"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df = df[df["text"].str.len() > 0].copy()
df = df[["text", "label"]].reset_index(drop=True)

print(f"Usable rows: {len(df):,}")


In [ ]:
def class_report(frame: pd.DataFrame, split_name: str = "full") -> None:
    counts = Counter(frame["label"].tolist())
    total  = len(frame)
    print(f"[{split_name}] n={total:,}")
    for lid in sorted(ID2LABEL):
        n   = counts.get(lid, 0)
        pct = 100 * n / total if total else 0
        print(f"{ID2LABEL[lid]:>10} ({lid})  {n:5,}  {pct:5.1f}%")

class_report(df, "full")

### 3.3 Stratified split — 90 / 5 / 5


In [ ]:
from collections import Counter

def class_report(frame: pd.DataFrame, split_name: str = "full") -> None:
    counts = Counter(frame["label"].tolist())
    total  = len(frame)
    print(f"[{split_name}] n={total:,}")
    for lid in sorted(ID2LABEL):
        n   = counts.get(lid, 0)
        pct = 100 * n / total if total else 0
        print(f"{ID2LABEL[lid]:>10} ({lid})  {n:5,}  {pct:5.1f}%")

# Shifted to a 90/5/5 split for the 160k-row dataset (~144k train, ~8k val, ~8k test)
train_df, temp_df = train_test_split(
    df, test_size=0.10, stratify=df["label"], random_state=RANDOM_SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=RANDOM_SEED
)

for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    split.to_csv(os.path.join(PROCESSED_DIR, f"{name}.csv"), index=False)
    print()
    class_report(split, name)

### 3.4 Class weights

Manual class weights used by the focal loss: normal = 1.00, flagged = 1.25.


In [ ]:
# Fixed manual weights for the binary classifier.
# Keep the same weighting for every experiment in this notebook.
CLASS_WEIGHTS = torch.tensor([1.0, 1.25], dtype=torch.float, device=DEVICE)
weight_dict = {ID2LABEL[i]: float(CLASS_WEIGHTS[i].item()) for i in range(NUM_LABELS)}

with open(os.path.join(PROCESSED_DIR, "class_weights.json"), "w") as f:
    json.dump(weight_dict, f, indent=2)

print(f"Class weights: {weight_dict}")


## 4. Dataset and loaders

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tokenizer):
        self.texts  = frame["text"].tolist()
        self.labels = frame["label"].tolist()
        self.tok    = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }


def make_loaders(tokenizer):
    frames = {}
    for name in ["train", "val", "test"]:
        f = pd.read_csv(os.path.join(PROCESSED_DIR, f"{name}.csv"))
        f["text"]  = f["text"].fillna("").astype(str)
        f["label"] = f["label"].astype(int)
        frames[name] = f

    kw = dict(batch_size=BATCH_SIZE, num_workers=2,
              pin_memory=(DEVICE.type == "cuda"))

    return (
        DataLoader(TweetDataset(frames["train"], tokenizer), shuffle=True,  **kw),
        DataLoader(TweetDataset(frames["val"],   tokenizer), shuffle=False, **kw),
        DataLoader(TweetDataset(frames["test"],  tokenizer), shuffle=False, **kw),
        frames["test"],
    )

In [ ]:
print("Loading BERTweet tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, use_fast=False, normalization=True
)

# Truncation diagnostic. BERTweet is trained around a 128-token input budget,
# while Jigsaw comments can be longer than tweets. Sample the training set so
# we can quantify how often content is being cut off without making startup slow.
diag_n = min(10000, len(train_df))
diag_texts = train_df["text"].sample(n=diag_n, random_state=RANDOM_SEED).tolist()
diag_lengths = np.asarray([
    len(tokenizer.encode(text, add_special_tokens=True, truncation=False))
    for text in diag_texts
])

print(f"Token-length diagnostic (sample n={diag_n:,}):")
print(f"  mean             : {diag_lengths.mean():.1f}")
print(f"  median           : {np.median(diag_lengths):.1f}")
print(f"  95th percentile  : {np.percentile(diag_lengths, 95):.1f}")
print(f"  > {MAX_LENGTH} tokens : {(diag_lengths > MAX_LENGTH).mean():.2%}")

train_loader, val_loader, test_loader, test_df_raw = make_loaders(tokenizer)
print(f"Batches — train: {len(train_loader)}  val: {len(val_loader)}  test: {len(test_loader)}")


## 5. Focal loss

The primary experiment uses focal loss with γ=1 and the fixed class weights above.


In [ ]:
class FocalLoss(nn.Module):
    """Multiclass focal loss — Lin et al. (2017).

    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    """
    def __init__(self, gamma: float = 1.0, alpha: torch.Tensor = None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce  = F.cross_entropy(logits, targets, weight=self.alpha, reduction="none")
        p_t = F.softmax(logits, dim=-1).gather(1, targets.unsqueeze(1)).squeeze(1)
        return ((1.0 - p_t) ** self.gamma * ce).mean()


loss_fn = FocalLoss(gamma=FOCAL_GAMMA, alpha=CLASS_WEIGHTS)
print(f"Focal loss ready (gamma={FOCAL_GAMMA})")


## 6. Train and eval loops

In [ ]:
def run_train_epoch(model, loader, optimizer, scheduler, loss_fn):
    model.train()
    total = 0.0
    for batch in loader:
        optimizer.zero_grad()
        logits = model(
            input_ids=batch["input_ids"].to(DEVICE),
            attention_mask=batch["attention_mask"].to(DEVICE),
        ).logits
        loss = loss_fn(logits, batch["labels"].to(DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total += loss.item()
    return total / len(loader)


def run_eval_probs(model, loader, loss_fn):
    """Return validation loss, flagged probabilities, and labels."""
    model.eval()
    total = 0.0
    probs_all, labs_all = [], []

    with torch.no_grad():
        for batch in loader:
            labels = batch["labels"].to(DEVICE)
            logits = model(
                input_ids=batch["input_ids"].to(DEVICE),
                attention_mask=batch["attention_mask"].to(DEVICE),
            ).logits

            total += loss_fn(logits, labels).item()
            probs = torch.softmax(logits, dim=-1)[:, 1]

            probs_all.extend(probs.detach().cpu().numpy())
            labs_all.extend(labels.detach().cpu().numpy())

    return total / len(loader), np.asarray(probs_all), np.asarray(labs_all)


def find_best_threshold(probs, labels, start=0.20, stop=0.80, step=0.005):
    """Choose the flagged threshold using validation Macro F1 only."""
    thresholds = np.arange(start, stop + step / 2, step)
    best_threshold = 0.50
    best_macro_f1 = -1.0

    for threshold in thresholds:
        preds = (probs >= threshold).astype(int)
        score = f1_score(labels, preds, average="macro", zero_division=0)
        if score > best_macro_f1:
            best_macro_f1 = float(score)
            best_threshold = float(threshold)

    return best_threshold, best_macro_f1


In [ ]:
def save_plot_history(history: dict, title: str, save_path: str):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="Train")
    axes[0].plot(history["val_loss"], label="Val")
    axes[0].set_title(f"{title} — Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history["val_macro_f1"], label="Threshold-tuned Val Macro F1")
    axes[1].set_title(f"{title} — Threshold-tuned Val Macro F1")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"Saved history plot: {save_path}")


def save_cm(labels, preds, title: str, save_path: str, cmap: str = "Greens"):
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap=cmap,
        xticklabels=list(ID2LABEL.values()),
        yticklabels=list(ID2LABEL.values()), ax=ax,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"Saved confusion matrix: {save_path}")


In [ ]:
def training_loop_cosine(model, train_loader, val_loader, loss_fn,
                         out_dir: str, lr: float, label: str,
                         num_epochs: int = NUM_EPOCHS,
                         patience: int = PATIENCE,
                         tokenizer=None):
    os.makedirs(out_dir, exist_ok=True)

    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=WEIGHT_DECAY,
    )

    total_steps = len(train_loader) * num_epochs
    warmup_steps = int(WARMUP_RATIO * total_steps)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    # IMPORTANT: checkpoint selection now uses the same kind of threshold-tuned
    # validation Macro F1 used by deployment instead of argmax / threshold=0.50.
    best_val_f1 = -1.0
    best_val_threshold = 0.50
    best_ckpt = os.path.join(out_dir, "best_model")
    patience_ctr = 0
    history = {
        "train_loss": [],
        "val_loss": [],
        "val_macro_f1": [],
        "val_threshold": [],
    }

    for epoch in range(1, num_epochs + 1):
        train_loss = run_train_epoch(model, train_loader, optimizer, scheduler, loss_fn)
        val_loss, val_probs, val_labels = run_eval_probs(model, val_loader, loss_fn)
        val_threshold, val_f1 = find_best_threshold(
            val_probs, val_labels, start=0.20, stop=0.80, step=0.005
        )

        history["train_loss"].append(float(train_loss))
        history["val_loss"].append(float(val_loss))
        history["val_macro_f1"].append(float(val_f1))
        history["val_threshold"].append(float(val_threshold))

        improved = val_f1 > best_val_f1
        marker = "  <- best" if improved else ""
        print(
            f"[{label}] Epoch {epoch:2d}/{num_epochs}  "
            f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
            f"val_macro_f1={val_f1:.4f}  threshold={val_threshold:.3f}{marker}"
        )

        if improved:
            best_val_f1 = float(val_f1)
            best_val_threshold = float(val_threshold)
            patience_ctr = 0
            if os.path.exists(best_ckpt):
                shutil.rmtree(best_ckpt)
            os.makedirs(best_ckpt, exist_ok=True)
            if tokenizer is not None:
                tokenizer.save_pretrained(best_ckpt)
            model.save_pretrained(best_ckpt)
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"[{label}] Early stopping at epoch {epoch} (patience={patience})")
                break

    save_plot_history(history, label, os.path.join(out_dir, "training_history.png"))
    print(f"\n[{label}] Best threshold-tuned Val Macro F1 = {best_val_f1:.4f}")
    print(f"[{label}] Best-epoch validation threshold = {best_val_threshold:.3f}")
    print(f"[{label}] Warmup steps = {warmup_steps:,} / {total_steps:,}")

    with open(os.path.join(out_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    return best_ckpt, best_val_f1, best_val_threshold


## 7. Model

LoRA uses query, key, value, and dense projection targets with r=32.


In [ ]:
def build_model():
    base = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )
    cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGETS,
        bias="none",
    )
    model = get_peft_model(base, cfg)
    model.print_trainable_parameters()
    return model


model = build_model().to(DEVICE)

## 8. Train

The best checkpoint is selected by **threshold-tuned validation Macro F1**, so model selection now matches the deployed decision rule rather than assuming a 0.50 threshold.


In [ ]:
LABEL = "BERTweet + LoRA r=32 + Focal(g=1) + Cosine Warmup + Tuned Checkpoint"

best_ckpt, best_val_f1, best_epoch_threshold = training_loop_cosine(
    model, train_loader, val_loader, loss_fn,
    OUT_DIR, LEARNING_RATE, LABEL,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    tokenizer=tokenizer,
)


## 9. Final validation threshold and held-out test evaluation

Reload the validation-selected checkpoint, refine its threshold using a finer 0.001 validation sweep, and only then evaluate once on the untouched test set.


In [ ]:
# Reload the checkpoint selected by threshold-tuned validation Macro F1.
base_for_eval = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID,
)
model_eval = PeftModel.from_pretrained(base_for_eval, best_ckpt).to(DEVICE)
model_eval.eval()

# Refine the selected checkpoint's threshold on validation data ONLY.
val_loss_final, val_probs, val_labels = run_eval_probs(model_eval, val_loader, loss_fn)
best_threshold, best_threshold_macro_f1 = find_best_threshold(
    val_probs,
    val_labels,
    start=0.20,
    stop=0.80,
    step=0.001,
)

print(f"Best validation threshold       : {best_threshold:.3f}")
print(f"Threshold-tuned Val Macro F1   : {best_threshold_macro_f1:.4f}")
print(f"Best checkpoint Val Macro F1   : {best_val_f1:.4f}")

with open(os.path.join(OUT_DIR, "threshold.json"), "w") as f:
    json.dump(
        {
            "threshold": best_threshold,
            "validation_macro_f1": best_threshold_macro_f1,
            "selection": "validation Macro F1 sweep from 0.20 to 0.80 in 0.001 increments",
        },
        f,
        indent=2,
    )

# The held-out test set is used only after checkpoint and threshold selection are complete.
_, test_probs, test_labels = run_eval_probs(model_eval, test_loader, loss_fn)
test_preds = (test_probs >= best_threshold).astype(int)

test_f1 = f1_score(test_labels, test_preds, average="macro", zero_division=0)
test_accuracy = accuracy_score(test_labels, test_preds)

report = classification_report(
    test_labels, test_preds,
    target_names=list(ID2LABEL.values()),
    output_dict=True, zero_division=0,
)

print(classification_report(
    test_labels, test_preds,
    target_names=list(ID2LABEL.values()), zero_division=0,
))

print(f"Selected Threshold        : {best_threshold:.3f}")
print(f"Test Macro F1             : {test_f1:.4f}")
print(f"Test Accuracy             : {test_accuracy:.4f}")
print(f"Best tuned Training Val F1: {best_val_f1:.4f}")


In [ ]:
save_cm(
    test_labels, test_preds,
    f"{LABEL} — Test (threshold={best_threshold:.3f})",
    os.path.join(OUT_DIR, "confusion_matrix.png"),
)

results = {
    "model": "bertweet_lora32_focal1_lr5e5_warmup_v4",
    "backbone": MODEL_NAME,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "lora_targets": LORA_TARGETS,
    "focal_gamma": FOCAL_GAMMA,
    "class_weights": [1.0, 1.25],
    "lr_schedule": "cosine_with_warmup",
    "warmup_ratio": WARMUP_RATIO,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "num_epochs": NUM_EPOCHS,
    "patience": PATIENCE,
    "checkpoint_selection": "threshold_tuned_validation_macro_f1",
    "best_epoch_threshold": round(float(best_epoch_threshold), 4),
    "threshold": round(float(best_threshold), 4),
    "threshold_selection": "validation_macro_f1_fine_sweep",
    "validation_threshold_macro_f1": round(float(best_threshold_macro_f1), 4),
    "test_macro_f1": round(float(test_f1), 4),
    "test_accuracy": round(float(test_accuracy), 4),
    "best_val_f1": round(float(best_val_f1), 4),
    "per_class": {
        c: {
            "precision": round(report[c]["precision"], 4),
            "recall": round(report[c]["recall"], 4),
            "f1": round(report[c]["f1-score"], 4),
        }
        for c in list(ID2LABEL.values()) if c in report
    },
}

with open(os.path.join(OUT_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))


## 10. Export a deployable artifact

The LoRA adapter alone cannot be served on its own — inference needs
the base model too. Merging the adapter weights back into the base
produces a single standard transformers directory that loads with
`AutoModelForSequenceClassification.from_pretrained(...)` and no PEFT
dependency at serving time.

Both are saved: the merged model for the FastAPI service, and the
adapter separately in case you want to keep training from it later.

In [ ]:
# Reload cleanly, then merge the adapter into the base weights
base_for_merge = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID,
)
peft_model = PeftModel.from_pretrained(base_for_merge, best_ckpt)

merged = peft_model.merge_and_unload()
merged.config.id2label = {int(k): v for k, v in ID2LABEL.items()}
merged.config.label2id = LABEL2ID

# merged = merged.half()

merged.save_pretrained(ARTIFACT_DIR)
tokenizer.save_pretrained(ARTIFACT_DIR)

print(f"Merged model written to {ARTIFACT_DIR}")

In [ ]:
# Copy the adapter alongside, for future continued training.
adapter_dir = os.path.join(ARTIFACT_DIR, "lora_adapter")
if os.path.exists(adapter_dir):
    shutil.rmtree(adapter_dir)
shutil.copytree(best_ckpt, adapter_dir)

# Metadata the serving layer can read without loading the model.
metadata = {
    "name": "openstream-moderation",
    "version": "v4",
    "task": "sequence-classification",
    "num_labels": NUM_LABELS,
    "id2label": ID2LABEL,
    "label2id": LABEL2ID,
    "max_length": MAX_LENGTH,
    "backbone": MODEL_NAME,
    "classification_threshold": float(best_threshold),
    "threshold_selection": "validation_macro_f1_fine_sweep",
    "checkpoint_selection": "threshold_tuned_validation_macro_f1",
    "training": {
        "method": "LoRA (merged)",
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "lora_targets": LORA_TARGETS,
        "loss": f"focal (gamma={FOCAL_GAMMA}, class-weighted)",
        "class_weights": [1.0, 1.25],
        "lr_schedule": "cosine with warmup",
        "warmup_ratio": WARMUP_RATIO,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "max_epochs": NUM_EPOCHS,
        "early_stopping_patience": PATIENCE,
    },
    "metrics": results,
    "preprocessing": "minimal whitespace cleanup; BERTweet tokenizer normalization=True handles model-specific normalization",
}

with open(os.path.join(ARTIFACT_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

with open(os.path.join(ARTIFACT_DIR, "threshold.json"), "w") as f:
    json.dump(
        {
            "threshold": float(best_threshold),
            "validation_macro_f1": float(best_threshold_macro_f1),
            "selection": "validation Macro F1 sweep from 0.20 to 0.80 in 0.001 increments",
        },
        f,
        indent=2,
    )

total = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, fs in os.walk(ARTIFACT_DIR) for f in fs
)
print(f"Artifact directory: {ARTIFACT_DIR}")
print(f"Total size: {total / 1e6:.1f} MB")
for f in sorted(os.listdir(ARTIFACT_DIR)):
    print(f"  {f}")


## 11. Verify the artifact loads and predicts

This is the exact code path the FastAPI service will use, so a clean
run here means the service will work.

In [ ]:
# Load purely from disk — no PEFT, no training state.
verify_tok = AutoTokenizer.from_pretrained(ARTIFACT_DIR, use_fast=False, normalization=True)
verify_model = AutoModelForSequenceClassification.from_pretrained(ARTIFACT_DIR).to(DEVICE)
verify_model.eval()

with open(os.path.join(ARTIFACT_DIR, "threshold.json"), "r") as f:
    verify_threshold = float(json.load(f)["threshold"])


def clean_tweet(text: str) -> str:
    # Match training preprocessing. BERTweet-specific normalization is performed
    # by verify_tok because it was loaded with normalization=True.
    if not isinstance(text, str):
        return ""
    return re.sub(r"\s+", " ", text).strip()


def classify(text: str) -> dict:
    cleaned = clean_tweet(text)
    enc = verify_tok(
        cleaned,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        logits = verify_model(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
        ).logits
        probs = F.softmax(logits, dim=-1).squeeze(0).cpu().numpy()

    flagged_prob = float(probs[1])
    pred = int(flagged_prob >= verify_threshold)
    threshold_margin = abs(flagged_prob - verify_threshold)

    return {
        "label": ID2LABEL[pred],
        "flagged_probability": round(flagged_prob, 4),
        "threshold": round(verify_threshold, 4),
        "threshold_margin": round(float(threshold_margin), 4),
        "margin_direction": "above" if flagged_prob >= verify_threshold else "below",
        "scores": {
            ID2LABEL[i]: round(float(probs[i]), 4)
            for i in range(NUM_LABELS)
        },
    }

samples = [
    "hello I am anutej",
    "I dont like real madriad",
    "I want to kill the person",
    "I am going to attend the protest against the governor",
]

for s in samples:
    r = classify(s)
    print(
        f"{r['label']:>10}  flagged={r['flagged_probability']:.4f}  "
        f"threshold={r['threshold']:.3f}  "
        f"margin={r['threshold_margin']:.4f} {r['margin_direction']}  |  {s}"
    )


### 11.5. Interactive Testing Environment

Use the text box below to test the model on your own custom inputs before downloading it.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

text_input = widgets.Textarea(
    value='',
    placeholder='Type a message to classify...',
    description='Input:',
    layout=widgets.Layout(width='80%', height='80px')
)

analyze_button = widgets.Button(
    description='Classify Text',
    button_style='primary',
    icon='search'
)

output_area = widgets.Output()


def on_analyze_clicked(b):
    with output_area:
        clear_output()
        text = text_input.value.strip()
        if not text:
            print("Please enter some text to classify.")
            return

        print(f"Analyzing: '{text}'\n")
        try:
            result = classify(text)
            label = result["label"].upper()
            flagged_prob = result["flagged_probability"]
            threshold = result["threshold"]
            margin = result["threshold_margin"]
            direction = result["margin_direction"]

            print(f"Prediction: {label}")
            print(f"Flagged probability : {flagged_prob:.4f}")
            print(f"Decision threshold  : {threshold:.4f}")
            print(f"Threshold margin    : {margin:.4f} {direction} threshold\n")
            print("All Class Scores:")
            for cls, score in result["scores"].items():
                print(f"  - {cls.ljust(10)}: {score:.4f}")
        except Exception as e:
            print(f"Error during classification: {e}")


analyze_button.on_click(on_analyze_clicked)
display(widgets.VBox([text_input, analyze_button, output_area]))


## 12. Package for download

The zip is what you copy into the Python moderation service.

In [ ]:
shutil.make_archive(ARCHIVE_NAME, "zip", ARTIFACT_DIR)
size = os.path.getsize(f"{ARCHIVE_NAME}.zip") / 1e6
print(f"{ARCHIVE_NAME}.zip  ({size:.1f} MB)")


In [ ]:
# Colab only
try:
    from google.colab import files
    files.download(f"{ARCHIVE_NAME}.zip")
except ImportError:
    print("Not running in Colab — download the zip from the file browser.")


---

## Notes for the serving layer

**Memory.** The merged model is BERTweet-base at fp32, roughly 540 MB on disk and similar in RAM. Render's free tier gives 512 MB, so it will not fit as-is. Two practical options are ONNX/int8 quantization or a service with more memory.

**Threshold.** The service must load `threshold.json` and compare the flagged probability against the saved validation-selected threshold. Do not use `argmax()` for the final hard decision.

**Preprocessing.** Keep deployment preprocessing identical to this notebook: only normalize whitespace before passing text to the BERTweet tokenizer loaded with `normalization=True`.
